# Cultural Economics — The Number Is the Boundary

Every other source in this corpus can be wrong about something it measured. The Arts and
Cultural Production Satellite Account cannot, because it does not measure anything. Nothing
here was sampled, so there is no margin of error. Nothing is withheld, so there are no
suppression flags. Nothing is perturbed, and the parts sum to the whole to the last dollar.

It is an **accounting re-partition**: the Bureau of Economic Analysis takes value added that
is already in the national accounts and re-sorts it under a published decision about which
industries — and which *shares* of which industries — constitute arts and cultural
production. The entire error budget of the statistic sits in that decision, and none of it is
published as a number.

This notebook measures how much work the boundary is doing. The answer is: almost all of it.
The headline figure is that arts and cultural production is a little over four per cent of
the US economy. Under a tenth of that comes from performing arts companies, museums,
independent artists, agents and fine-arts education combined. And the state with the highest
published arts location quotient in the country turns out to be the state whose arts economy
has the least arts in it.

## Learning Objectives

1. **Recognise a satellite account** and understand why its perfect internal consistency is a
   property of accounting rather than evidence of accuracy.
2. **Locate a statistic's error budget** when no uncertainty is published, by varying the
   definition instead of the data.
3. **Refuse to sum a column** whose rows are a grand total, a subject total, subtotals and
   leaves with no depth marker between them.
4. **Test a ranking against its own construct** rather than against sampling noise, which is
   the only test available when there is no sampling noise.
5. **Handle a source that echoes your credential back to you**, and see why the provenance
   hash has to be taken over the stripped payload.

## Prerequisites

- Handbook §1.2 (BLS LAUS — model output with no published uncertainty). BEA sits beside it
  and goes further: LAUS at least predicts a quantity that could in principle be counted.
- Handbook §8.3 (construct validity is not precision). This notebook is that section's
  worked case.
- Handbook §7 on what a ranking may claim. Here there is no interval to reason with at all,
  so the machinery in §7.1 cannot even be applied — §3 below builds the substitute.

## Data Sources

| endpoint | table | what a row is |
|---|---|---|
| `apps.bea.gov/api/data` (Regional) | `SAACArtsVA` | value added, one geography × ACPSA line × year |
| | `SAACVALQ` | the published arts value-added location quotient |

Reference year **2023**, the latest published. The Regional endpoint takes four parameters
and none of them is a vintage: the only temporal control is a `Year` window whose default,
`LAST5`, moves with each annual release. `fetch_bea_regional` requires an explicit year for
that reason.

**A note on the credential.** BEA is the only source in this corpus that returns the
submitted key inside the response body, at `BEAAPI.Request.RequestParam`. The client removes
that block before parsing, so it reaches neither this notebook's stored output nor the
provenance record — and the payload hash is taken over the stripped document, so another
analyst with a different key reproduces the same hash.

In [1]:
"""Configuration. Every choice that could drift is named here."""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy import stats

import sys
from pathlib import Path

#: Find the package by walking UP from wherever this runs, so the notebook works
#: whether it is executed in place or from the repository root. Sixteen notebooks
#: imported `krl_tutorials_data` with no path discovery and therefore ran only
#: from the root -- which nobody noticed while every run happened to start there.
_repo = next((c for c in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
              if (c / "src" / "krl_tutorials_data").is_dir()), None)
if _repo is None:
    raise SystemExit("krl_tutorials_data not found above " + str(Path.cwd()))
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo / "src"))

from krl_tutorials_data.sources import (
    verify_digest,
    BEA_ACPSA_LINE_ROLES,
    BEA_ACPSA_TABLES,
    fetch_bea_regional,
)

YEAR = 2023                  # explicit; BEA's own default is a moving five-year window
NATION = "00000"             # the United States row. "US" is accepted and returns nothing.
STATES = "STATE"

#: The three leaf industries that dominate the account and that almost nobody
#: means by "the arts". Named here so §3 can remove them by name rather than by
#: position, and so the choice is visible rather than buried in a slice.
LARGEST_NON_ARTS_LINES = {
    "225": "Other information services",
    "224": "Broadcasting",
    "214": "Government",
}

#: Lines a lay reader would recognise as arts organisations and artists.
RECOGNISABLY_ARTS_LINES = {
    "111": "Performing arts companies",
    "112": "Promoters of performing arts and similar events",
    "113": "Agents/managers for artists",
    "114": "Independent artists, writers, and performers",
    "120": "Museums",
    "140": "Fine arts education",
}

pd.set_option("display.width", 118)
pd.set_option("display.max_rows", 60)

print(f"reference year : {YEAR}")
print(f"ACPSA tables   : {sorted(BEA_ACPSA_TABLES)}")
print("line roles the API does not express:")
for line, role in BEA_ACPSA_LINE_ROLES.items():
    print(f"  {line:>4s}  {role}")

#: The digest of the payload every figure below was computed from, on this
#: record's own hash basis. BEA serves no prior vintage -- 0 of 13 datasets offer
#: an as-of parameter -- so this pin does not restore one. It guarantees an
#: upstream revision cannot pass unnoticed: the notebook raises instead of
#: reporting different numbers under unchanged prose. Handbook §5.2.
PINNED_BEA_NATIONAL = "322bc3e22952f2b444d98ee448e319a8ab4a2ee5420bf209b9970b5c18cfb0b7"


reference year : 2023
ACPSA tables   : ['SAACArtsComp', 'SAACArtsEmp', 'SAACArtsVA', 'SAACCompLQ', 'SAACCompRatio', 'SAACVALQ', 'SAACVARatio']
line roles the API does not express:
     1  grand total -- TOTAL VALUE ADDED OF THE WHOLE ECONOMY, not an arts line
    10  subject total -- total ACPSA value added
   100  subtotal -- core arts and cultural production
   200  subtotal -- supporting arts and cultural production
   300  subtotal -- all other industries


In [2]:
"""Retrieve every frame this notebook uses."""
national, prov_nat = verify_digest(
    fetch_bea_regional(
    table="SAACArtsVA", line_code="ALL", year=YEAR, geo_fips=NATION
),
    PINNED_BEA_NATIONAL, label='BEA ACPSA SAACArtsVA, national 2023')

def state_series(line_code: str) -> pd.Series:
    """One ACPSA line across the 51 state rows, indexed by state name."""
    frame, _ = fetch_bea_regional(
        table="SAACArtsVA", line_code=line_code, year=YEAR, geo_fips=STATES
    )
    return frame.set_index("GeoName")["DataValue"]

total_value_added = state_series("1")     # the whole economy, NOT an arts line
acpsa_total = state_series("10")
core_arts = state_series("100")
supporting = state_series("200")
all_other = state_series("300")

published_lq, _ = fetch_bea_regional(
    table="SAACVALQ", line_code="10", year=YEAR, geo_fips=STATES
)
published_lq = published_lq.set_index("GeoName")["DataValue"]

print(f"national rows (all lines)  : {len(national)}")
print(f"state rows per line        : {len(acpsa_total)}")
print(f"published location quotients: {len(published_lq)}  unit as served: "
      f"{BEA_ACPSA_TABLES['SAACVALQ']}")
print()
for note in prov_nat.notes:
    print(f"  - {note}")

national rows (all lines)  : 39
state rows per line        : 51
published location quotients: 51  unit as served: Arts value added location quotient

  - BEA Regional table SAACArtsVA -- ACPSA value added by ACPSA industry. Units: Thousands of dollars.
  - SATELLITE ACCOUNT. These are not counts, survey estimates or model predictions of an observable quantity: they are an accounting re-partition of value added under a published boundary decision about which industries, and which shares of them, constitute arts and cultural production. No margin of error exists because nothing was sampled. The uncertainty is in the boundary and is not published.
  - The submitted credential is echoed by BEA at BEAAPI.Request and was removed before parsing; the payload hash below is over the stripped document, so another analyst using a different key reproduces it.
  - Vintage pinned to 2023. The Regional endpoint exposes no vintage parameter -- only a Year window whose default, 'LAST5', moves with each 

## 1. What is inside the boundary

The account has thirty-nine lines. Two of them are totals of different things, three are
subtotals, and thirty-four are industries. Ranking the industries by how much of the arts
total they supply is the whole argument of this notebook, and it takes one sort.

In [3]:
"""The composition of arts and cultural production, as published."""
nat = national.set_index("LineCode")
value = nat["DataValue"]
label = (
    nat["Description"].astype("string")
    .str.replace(r"^\[SAACArtsVA\]\s*(ACPSA Value Added:\s*)?", "", regex=True)
    .str.strip()
)

economy = float(value["1"])
acpsa = float(value["10"])

leaves = nat[nat["LineRole"] == "leaf"].copy()
leaves["share_of_acpsa"] = leaves["DataValue"] / acpsa
leaves["label"] = label.reindex(leaves.index)
ranked = leaves.sort_values("share_of_acpsa", ascending=False)

print(f"Total value added of the US economy (line 1) : ${economy / 1e6:>10,.1f}B")
print(f"Total ACPSA value added (line 10)            : ${acpsa / 1e6:>10,.1f}B")
print(f"ACPSA as a share of the economy              : {acpsa / economy:>10.2%}")
print()
print(f"the {len(ranked)} industries, by share of the arts total:")
view = ranked[["label", "DataValue", "share_of_acpsa"]].head(16)
print(view.to_string(
    index=True, header=["industry", "value added ($k)", "share"],
    formatters={"DataValue": "{:,.0f}".format, "share_of_acpsa": "{:.1%}".format},
))

recognisable = sum(float(value[line]) for line in RECOGNISABLY_ARTS_LINES)
biggest_three = sum(float(value[line]) for line in LARGEST_NON_ARTS_LINES)
print()
print("grouping the same lines two ways:")
print(f"  {', '.join(RECOGNISABLY_ARTS_LINES.values())[:88]}...")
print(f"    combined: ${recognisable / 1e6:,.1f}B = {recognisable / acpsa:.1%} of ACPSA")
print(f"  {', '.join(LARGEST_NON_ARTS_LINES.values())}")
print(f"    combined: ${biggest_three / 1e6:,.1f}B = {biggest_three / acpsa:.1%} of ACPSA")
print()
print(f"  core arts subtotal (line 100)       : {float(value['100']) / acpsa:.1%}")
print(f"  supporting subtotal (line 200)      : {float(value['200']) / acpsa:.1%}")
print(f"  all other industries (line 300)     : {float(value['300']) / acpsa:.1%}")

Total value added of the US economy (line 1) : $  27,567.3B
Total ACPSA value added (line 10)            : $   1,169.8B
ACPSA as a share of the economy              :      4.24%

the 34 industries, by share of the arts total:
                                                 industry value added ($k)  share
LineCode                                                                         
225                            Other information services      181,537,690  15.5%
224                                          Broadcasting      148,800,318  12.7%
214                                            Government      135,278,786  11.6%
221                                            Publishing      125,474,797  10.7%
222                                       Motion pictures       90,678,414   7.8%
260                                     Retail industries       66,001,790   5.6%
131                                           Advertising       56,875,691   4.9%
250               Wholesale and tran

The word "arts" is carrying a great deal of freight. This is not an error in the account —
BEA documents every inclusion, and the reasoning is defensible: a graphic designer at an
advertising agency is producing cultural output whether or not anyone calls it art, and a
broadcaster distributes it. The point is narrower and it is about *reading*, not about BEA.

**A reader who hears "arts and culture is four per cent of the economy" and pictures theatres
and museums has misread the statistic by roughly an order of magnitude**, and nothing in the
published table corrects them, because the table's own name is the claim.

## 2. The column you must not sum

Before going further, a mechanical trap that costs more than it should. The thirty-nine lines
arrive as sibling rows. There is no depth column, no indentation, no parent identifier. The
grand total of the entire US economy, the arts total, three subtotals and thirty-four
industries are all just rows.

In [4]:
"""What happens if you treat the line codes as a flat list."""
naive = float(national["DataValue"].sum())

print(f"rows returned                          : {len(national)}")
print(f"naive sum of the DataValue column      : ${naive / 1e6:>10,.1f}B")
print(f"the true ACPSA total (line 10)         : ${acpsa / 1e6:>10,.1f}B")
print(f"the naive sum is                       : {naive / acpsa:>10.2f}x too large")
print(f"of which line 1 alone contributes      : ${economy / 1e6:>10,.1f}B "
      f"({economy / naive:.1%} of the naive sum)")
print()
print("what each role contributes to that mistake:")
by_role = national.groupby("LineRole")["DataValue"].agg(["size", "sum"])
print(by_role.to_string(header=["rows", "value added ($k)"],
                        formatters={"sum": "{:,.0f}".format}))
subtotals = sum(float(value[line]) for line in ("100", "200", "300"))
leaf_sum = float(leaves["DataValue"].sum())
residual = float(value["300"])
children = {parent: sum(1 for k in leaves.index if k.startswith(parent[0]))
            for parent in ("100", "200", "300")}

print()
print("the additive statements that ARE true:")
print(f"  100 + 200 + 300            = ${subtotals / 1e6:>8,.1f}B  vs line 10 "
      f"${acpsa / 1e6:,.1f}B   gap {abs(subtotals - acpsa) / acpsa:.7%}")
print(f"  {len(leaves)} leaves + line 300       = ${(leaf_sum + residual) / 1e6:>8,.1f}B  vs line 10 "
      f"${acpsa / 1e6:,.1f}B   gap {abs(leaf_sum + residual - acpsa) / acpsa:.7%}")
print(f"  the {len(acpsa_total)} states summed       = ${float(acpsa_total.sum()) / 1e6:>8,.1f}B  vs the US row "
      f"${acpsa / 1e6:,.1f}B   gap {abs(float(acpsa_total.sum()) - acpsa) / acpsa:.7%}")

print()
print("and one that looks true and is not:")
print(f"  the {len(leaves)} leaves alone          = ${leaf_sum / 1e6:>8,.1f}B  vs line 10 "
      f"${acpsa / 1e6:,.1f}B   SHORT BY {(acpsa - leaf_sum) / acpsa:.4%}")
print(f"  leaf industries beneath each subtotal: "
      + ", ".join(f"{parent}={count}" for parent, count in children.items()))
print(f"  '{label['300']}' is a residual of ${residual / 1e6:,.1f}B "
      f"that belongs to no named industry line.")

rows returned                          : 39
naive sum of the DataValue column      : $  31,046.4B
the true ACPSA total (line 10)         : $   1,169.8B
the naive sum is                       :      26.54x too large
of which line 1 alone contributes      : $  27,567.3B (88.8% of the naive sum)

what each role contributes to that mistake:
                                                                        rows value added ($k)
LineRole                                                                                     
grand total -- TOTAL VALUE ADDED OF THE WHOLE ECONOMY, not an arts line    1   27,567,298,000
leaf                                                                      34    1,139,572,689
subject total -- total ACPSA value added                                   1    1,169,753,742
subtotal -- all other industries                                           1       30,181,057
subtotal -- core arts and cultural production                              1      265,805,188
sub

Two things in that output matter more than the twenty-six.

**The sums that should balance, balance to the seventh decimal.** Subtotals to their total,
states to the nation. That combination is characteristic of an accounting product and it
inverts the usual heuristic: in a survey or a register, ragged totals are a warning sign,
whereas here **exactness is guaranteed by construction and therefore carries no information
about whether the numbers are right.** The account would balance just as perfectly under a
completely different definition of the arts.

**And one sum that looks like it should balance does not.** Adding the named industries gives
less than the total, because `all other industries` is a residual subtotal with nothing
beneath it — value added attributed to arts and cultural production in industries the account
does not name individually. An analyst who adds the industry rows to check their work finds a
shortfall and will reasonably suspect rounding or a dropped row. It is neither. It is a
category whose contents are, by design, not enumerated.

## 3. If the error is in the boundary, vary the boundary

There is no margin to propagate, no reliability to compute, no vintage to compare against.
The standard toolkit for asking "how much does this number know" needs an uncertainty
statement and there is not one.

So use the one degree of freedom the account actually has. The published statistic is a
location quotient — a state's arts share of its own economy, divided by the nation's — and it
is fully reconstructible from the value-added table. Rebuild it under alternative boundaries
and see whether the answer survives. If the ranking is stable across reasonable definitions,
the boundary is not doing much work. If it is not, the boundary *is* the statistic.

In [5]:
"""Four boundaries, one location quotient."""
def location_quotient(numerator: pd.Series, national_numerator: float) -> pd.Series:
    """State arts share of its own economy, relative to the national share."""
    return (numerator / total_value_added) / (national_numerator / economy)

trimmed = acpsa_total - sum(state_series(line) for line in LARGEST_NON_ARTS_LINES)
national_trimmed = acpsa - sum(float(value[line]) for line in LARGEST_NON_ARTS_LINES)

boundaries = pd.DataFrame({
    "published (all ACPSA)": location_quotient(acpsa_total, acpsa),
    "core arts only": location_quotient(core_arts, float(value["100"])),
    "supporting only": location_quotient(supporting, float(value["200"])),
    "less the largest three": location_quotient(trimmed, national_trimmed),
})

reference = "published (all ACPSA)"
check = (boundaries[reference] - published_lq).abs()
print(f"reconstruction against BEA's published location quotient, n={len(check)}")
print(f"  median |error| = {check.median():.5f}   max = {check.max():.5f}   "
      f"within 0.005: {int((check < 0.005).sum())}/{len(check)}")
print("  (the residual is publication rounding; the formula is confirmed, not assumed)")
print()

ranks = boundaries.rank(ascending=False)
print(f"how much the state ordering moves when the boundary moves, n={len(boundaries)}")
for column in boundaries.columns:
    if column == reference:
        continue
    rho, p_value = stats.spearmanr(boundaries[reference], boundaries[column])
    moved = (ranks[reference] - ranks[column]).abs()
    print(f"  {column:24s} rho={rho:+.3f} (p={p_value:.1e})  median move={moved.median():>4.0f} "
          f"places  max={moved.max():>3.0f}  moved >10: {int((moved > 10).sum())}/{len(moved)}")

reconstruction against BEA's published location quotient, n=51
  median |error| = 0.00380   max = 0.01263   within 0.005: 42/51
  (the residual is publication rounding; the formula is confirmed, not assumed)

how much the state ordering moves when the boundary moves, n=51
  core arts only           rho=+0.852 (p=2.3e-15)  median move=   4 places  max= 31  moved >10: 7/51
  supporting only          rho=+0.978 (p=3.1e-35)  median move=   2 places  max=  9  moved >10: 0/51
  less the largest three   rho=+0.937 (p=4.5e-24)  median move=   3 places  max= 14  moved >10: 2/51


In [6]:
"""Who is in the top eight, and under which definition."""
TOP = 8
tops = {c: list(boundaries[c].sort_values(ascending=False).head(TOP).index)
        for c in boundaries.columns}
for column, names in tops.items():
    print(f"  {column:24s} {', '.join(names)}")

published_top = set(tops[reference])
core_top = set(tops["core arts only"])
print()
print(f"  overlap between the published top {TOP} and a core-arts top {TOP}: "
      f"{len(published_top & core_top)}/{TOP}")
print(f"    leave on a core-arts boundary : {sorted(published_top - core_top)}")
print(f"    enter on a core-arts boundary : {sorted(core_top - published_top)}")

  published (all ACPSA)    Washington, District of Columbia, New York, California, Nevada, Tennessee, Massachusetts, Connecticut
  core arts only           District of Columbia, New York, California, Nevada, Tennessee, Colorado, Massachusetts, Illinois
  supporting only          Washington, District of Columbia, California, New York, Nevada, Connecticut, Massachusetts, Tennessee
  less the largest three   Washington, California, New York, District of Columbia, Tennessee, Nevada, Massachusetts, Utah

  overlap between the published top 8 and a core-arts top 8: 6/8
    leave on a core-arts boundary : ['Connecticut', 'Washington']
    enter on a core-arts boundary : ['Colorado', 'Illinois']


A rank correlation of this size is the kind of number that usually reassures. It should not
here, and the reason is in the second row of that table: the *supporting industries* boundary
tracks the published one almost perfectly, because supporting industries are most of the
published total. **The headline arts location quotient is very nearly a supporting-industries
location quotient wearing an arts label**, and it agrees with a core-arts measure a good deal
less well than it agrees with itself.

The individual movements are where this becomes concrete.

In [7]:
"""The same location quotient over very different economies."""
composition = pd.DataFrame({
    "published LQ": boundaries[reference],
    "core-arts LQ": boundaries["core arts only"],
    "core share of its own ACPSA": core_arts / acpsa_total,
    "rank (published)": ranks[reference],
    "rank (core arts)": ranks["core arts only"],
})
composition["rank move"] = composition["rank (published)"] - composition["rank (core arts)"]

national_core_share = float(value["100"]) / acpsa
print(f"core arts as a share of each state's own ACPSA value added, {len(composition)} states")
print(f"  national  : {national_core_share:.1%}")
print(f"  min       : {composition['core share of its own ACPSA'].min():.1%} "
      f"({composition['core share of its own ACPSA'].idxmin()})")
print(f"  median    : {composition['core share of its own ACPSA'].median():.1%}")
print(f"  max       : {composition['core share of its own ACPSA'].max():.1%} "
      f"({composition['core share of its own ACPSA'].idxmax()})")
print()
print(f"the published top {TOP}, with what is actually in their arts economies:")
top_view = composition.loc[tops[reference]]
print(top_view.to_string(formatters={
    "published LQ": "{:.2f}".format, "core-arts LQ": "{:.2f}".format,
    "core share of its own ACPSA": "{:.1%}".format,
    "rank (published)": "{:.0f}".format, "rank (core arts)": "{:.0f}".format,
    "rank move": "{:+.0f}".format}))
print()
biggest = composition["rank move"].abs().idxmax()
print(f"largest single movement: {biggest}, "
      f"rank {composition.loc[biggest, 'rank (published)']:.0f} on the published boundary and "
      f"{composition.loc[biggest, 'rank (core arts)']:.0f} on a core-arts one")

core arts as a share of each state's own ACPSA value added, 51 states
  national  : 22.7%
  min       : 5.8% (Washington)
  median    : 22.7%
  max       : 31.7% (Vermont)

the published top 8, with what is actually in their arts economies:
                     published LQ core-arts LQ core share of its own ACPSA rank (published) rank (core arts) rank move
GeoName                                                                                                               
Washington                   2.30         0.59                        5.8%                1               32       -31
District of Columbia         2.01         2.70                       30.5%                2                1        +1
New York                     1.79         1.99                       25.3%                3                2        +1
California                   1.76         1.76                       22.8%                4                3        +1
Nevada                       1.19         1.6

## 4. The vintage you cannot cite, and the uncertainty that is not there

Two structural properties remain, and both are absences rather than values.

**There is no vintage.** The Regional endpoint takes four parameters — `GeoFips`, `LineCode`,
`TableName`, `Year` — and none of them selects a published vintage. `Year` is not even
required: its default is `LAST5`, a window that moves with every annual release. A notebook
that omits it returns different years next year and raises nothing. Whether and how much BEA
revises these estimates is not established here — and that is precisely the problem. The API
serves one vintage and offers no way to ask for another, so a revision, if it happened, is
invisible from this interface. That puts this source with every other in the corpus except
FRED (handbook §5.5).

In [8]:
"""The client refuses the moving default, and says why."""
for attempt in ("LAST5", "ALL", "2019-2023"):
    try:
        fetch_bea_regional(table="SAACArtsVA", line_code="10", year=attempt, geo_fips=NATION)
    except ValueError as refusal:
        print(f"  year={attempt!r:12s} -> refused: {refusal}")
    else:
        print(f"  year={attempt!r:12s} -> ACCEPTED, which this notebook's premise says it "
              f"should not be")

print()
print("what the moving window would have covered, had it been used:")
span = {}
for year in (2019, 2020, 2021, 2022, YEAR):
    frame, _ = fetch_bea_regional(
        table="SAACArtsVA", line_code="ALL", year=year, geo_fips=NATION
    )
    indexed = frame.set_index("LineCode")["DataValue"]
    span[year] = float(indexed["10"]) / float(indexed["1"])

series = pd.Series(span, name="ACPSA share of value added")
print(series.to_string(float_format="{:.3%}".format))
print()
print(f"  range across those {len(series)} years: "
      f"{series.min():.3%} to {series.max():.3%}, "
      f"a spread of {(series.max() - series.min()) * 100:.2f} percentage points")

  year='LAST5'      -> refused: year must be a four-digit year, not 'LAST5'. BEA's own default is 'LAST5', a moving window; a stored artifact cannot cite it.
  year='ALL'        -> refused: year must be a four-digit year, not 'ALL'. BEA's own default is 'LAST5', a moving window; a stored artifact cannot cite it.
  year='2019-2023'  -> refused: year must be a four-digit year, not '2019-2023'. BEA's own default is 'LAST5', a moving window; a stored artifact cannot cite it.

what the moving window would have covered, had it been used:


2019   4.348%
2020   4.333%
2021   4.339%
2022   4.204%
2023   4.243%

  range across those 5 years: 4.204% to 4.348%, a spread of 0.14 percentage points


**And there is no uncertainty statement of any kind.** Not a margin, not a credible interval,
not a standard error, not a suppression flag. Checked across the eight smallest ACPSA
industries at state level, every cell is a number.

This places BEA alongside BLS LAUS in handbook §1.2 — published estimates with no published
uncertainty — but it is a step further along. LAUS is a model prediction of a quantity that
could in principle be counted: someone either was or was not unemployed. "Arts and cultural
value added" is not a quantity that exists prior to the accounting decision that defines it.
There is nothing it could be measured against, so the absence of a margin is not an omission.
It is the correct representation of a number that has no sampling error and unbounded
definitional error.

**[rule]** When a statistic has no uncertainty because it was not measured, do not report it
as though it were precise. Report the sensitivity of your conclusion to the definition, which
is the only error bar available — and which §3 shows is not small.

In [9]:
"""FINDINGS"""
findings = []

findings.append((
    "F1",
    f"ACPSA value added is ${acpsa / 1e6:,.1f}B, {acpsa / economy:.2%} of US value added in "
    f"{YEAR}. Performing arts companies, promoters, agents, independent artists, museums and "
    f"fine-arts education together supply {recognisable / acpsa:.1%} of it."
))
findings.append((
    "F2",
    f"The three largest lines that no lay reader would call arts -- "
    f"{', '.join(LARGEST_NON_ARTS_LINES.values())} -- supply {biggest_three / acpsa:.1%}, "
    f"and the single largest industry in the account is "
    f"{ranked['label'].iloc[0]} at {ranked['share_of_acpsa'].iloc[0]:.1%}."
))
findings.append((
    "F3",
    f"Core arts is {float(value['100']) / acpsa:.1%} of the account, supporting production "
    f"{float(value['200']) / acpsa:.1%}, all other industries "
    f"{float(value['300']) / acpsa:.1%}."
))
findings.append((
    "F4",
    f"The line codes are not a flat list. Summing the DataValue column of the "
    f"{len(national)}-row national response gives {naive / acpsa:.2f}x the true ACPSA total, "
    f"because line 1 is the whole economy rather than an arts line."
))
findings.append((
    "F5",
    f"The sums that should balance do, to the seventh decimal: subtotals to total within "
    f"{abs(subtotals - acpsa) / acpsa:.7%}, the {len(leaves)} leaves plus the residual line "
    f"within {abs(leaf_sum + residual - acpsa) / acpsa:.7%}, and the {len(acpsa_total)} state "
    f"rows to the US row within {abs(float(acpsa_total.sum()) - acpsa) / acpsa:.7%}. "
    f"Exactness is a property of accounting, not evidence of accuracy."
))
findings.append((
    "F5b",
    f"The named industries are not a partition of the total: the {len(leaves)} leaves sum to "
    f"{(acpsa - leaf_sum) / acpsa:.4%} less than line 10, because 'all other industries' is a "
    f"residual subtotal of ${residual / 1e6:,.1f}B with no leaf lines beneath it. A reader who "
    f"adds the industry rows to check their work finds a shortfall that is not an error."
))
findings.append((
    "F6",
    f"The published location quotient is reconstructible from the value-added table: median "
    f"|error| {check.median():.5f}, max {check.max():.5f}, "
    f"{int((check < 0.005).sum())} of {len(check)} within 0.005."
))
_rho_core, _p_core = stats.spearmanr(boundaries[reference], boundaries["core arts only"])
_rho_sup, _ = stats.spearmanr(boundaries[reference], boundaries["supporting only"])
_moved_core = (ranks[reference] - ranks["core arts only"]).abs()
findings.append((
    "F7",
    f"Moving the boundary moves the ranking. Against a core-arts definition the published "
    f"LQ has Spearman {_rho_core:+.3f} (p={_p_core:.1e}), median move "
    f"{_moved_core.median():.0f} places, max {_moved_core.max():.0f}, with "
    f"{int((_moved_core > 10).sum())} of {len(_moved_core)} states moving more than ten."
))
findings.append((
    "F8",
    f"The published measure tracks a supporting-industries measure ({_rho_sup:+.3f}) more "
    f"closely than a core-arts one ({_rho_core:+.3f}): the headline arts statistic is very "
    f"nearly a supporting-industries statistic."
))
findings.append((
    "F9",
    f"Core arts ranges from {composition['core share of its own ACPSA'].min():.1%} "
    f"({composition['core share of its own ACPSA'].idxmin()}) to "
    f"{composition['core share of its own ACPSA'].max():.1%} "
    f"({composition['core share of its own ACPSA'].idxmax()}) of a state's own ACPSA value "
    f"added, national {national_core_share:.1%}. The same published LQ therefore describes "
    f"economies with very different arts content."
))
findings.append((
    "F10",
    f"Only {len(published_top & core_top)} of the published top {TOP} states remain in the "
    f"top {TOP} under a core-arts boundary; {sorted(published_top - core_top)} leave and "
    f"{sorted(core_top - published_top)} enter."
))
findings.append((
    "F11",
    f"No vintage is selectable: the Regional endpoint exposes four parameters and none of "
    f"them chooses a published vintage, while Year is optional and defaults to a moving "
    f"five-year window, so any revision is invisible from this interface. Over "
    f"{min(series.index)}-{max(series.index)} the published ACPSA share ranged "
    f"{series.min():.3%} to {series.max():.3%}, a spread of "
    f"{(series.max() - series.min()) * 100:.2f} percentage points."
))
findings.append((
    "F12",
    f"The credential BEA echoes at BEAAPI.Request was stripped before parsing; the provenance "
    f"hash is over the stripped document, so it is reproducible by an analyst holding a "
    f"different key."
))

for tag, text in findings:
    print(f"[{tag}] {text}")
    print()

[F1] ACPSA value added is $1,169.8B, 4.24% of US value added in 2023. Performing arts companies, promoters, agents, independent artists, museums and fine-arts education together supply 9.6% of it.

[F2] The three largest lines that no lay reader would call arts -- Other information services, Broadcasting, Government -- supply 39.8%, and the single largest industry in the account is Other information services at 15.5%.

[F3] Core arts is 22.7% of the account, supporting production 74.7%, all other industries 2.6%.

[F4] The line codes are not a flat list. Summing the DataValue column of the 39-row national response gives 26.54x the true ACPSA total, because line 1 is the whole economy rather than an arts line.

[F5] The sums that should balance do, to the seventh decimal: subtotals to total within 0.0000000%, the 34 leaves plus the residual line within 0.0000003%, and the 51 state rows to the US row within 0.0000003%. Exactness is a property of accounting, not evidence of accuracy.

[F5

## Key Takeaways

1. **A satellite account does not measure; it re-sorts.** Its internal consistency is
   guaranteed by construction and tells you nothing about whether the boundary is right. The
   account would balance just as perfectly under a completely different definition.

2. **The error budget lives in the definition, and you can measure it.** When no uncertainty
   is published, vary the boundary rather than the data. That is not a substitute for an error
   bar — it is the only error bar this class of statistic has.

3. **Read the name as a claim, not a label.** "Arts and cultural production" is doing
   enormous work in this account, and the gap between what the phrase suggests and what the
   number contains is roughly an order of magnitude.

4. **Never sum a column whose rows include their own totals.** BEA serves a grand total, a
   subject total, three subtotals and thirty-four leaves as siblings with no depth marker.
   `fetch_bea_regional` adds a `LineRole` column for exactly this reason.

5. **`LineCode` 1 is not the first arts line.** It is total value added of the whole economy.

6. **Pin the year.** BEA's default is a window that moves with each release, and no past
   vintage is retrievable, so an unpinned call is unreproducible in a way that raises nothing.

7. **A source can hand your credential back to you.** Strip it from the document before
   anything parses it, and hash the stripped payload — otherwise the provenance record is one
   that nobody else can reproduce.

## Exercises

1. **Build the boundary that matches your question.** Pick a policy question — arts
   employment, cultural tourism, creative-sector wages — and decide which of the thirty-four
   lines belong. Recompute the state ranking. Then state what would have to be true for
   BEA's boundary to be the right one for that question instead of yours.

2. **Test the account against an independent measure.** `SAACArtsEmp` gives ACPSA employment.
   Does a state's employment-based rank agree with its value-added rank? Where it does not,
   say which lines drive the disagreement and whether that is a data problem or a real
   difference between labour-intensive and capital-intensive cultural production.

3. **Quantify the revision you cannot see.** The API serves one vintage. Find BEA's published
   revision notes for ACPSA, and state — with a number — how large a revision would have to be
   before a conclusion in §3 changed. You will have to define "changed" first.

4. **Re-run §3 for compensation.** `SAACCompLQ` is the compensation location quotient. Is the
   boundary sensitivity larger or smaller than for value added, and does the explanation
   follow from which industries are labour-intensive?

5. **Write the caption.** In one sentence that a newspaper could print, state your state's
   arts location quotient in a way that is true. It must survive a reader who then looks up
   what is inside the account.

## References

- U.S. Bureau of Economic Analysis. *Arts and Cultural Production Satellite Account (ACPSA)*,
  Regional dataset tables `SAACArtsVA`, `SAACArtsComp`, `SAACArtsEmp`, `SAACVALQ`,
  `SAACCompLQ`, `SAACVARatio`, `SAACCompRatio`. `apps.bea.gov/api/data`.
- U.S. Bureau of Economic Analysis and National Endowment for the Arts. *ACPSA methodology* —
  the industry boundary, and the treatment of industries only partly attributable to arts and
  cultural production.
- U.S. Bureau of Economic Analysis. *BEA Data API user guide* — the Regional dataset's four
  parameters, the `LAST5` default, and the HTTP 200 error envelope.
- Handbook §1.2 (LAUS — model output with no published uncertainty), §5.5 (retrievability of
  past vintages), §8.3 (construct validity is not precision), §7.1 (what a ranking may claim).

## Limitations

- **One year.** Every cross-sectional figure is 2023. §4 shows the national share across five
  years; the boundary-sensitivity result in §3 is not recomputed for other years, and nothing
  here establishes that the rank movements are stable over time.
- **The alternative boundaries are illustrative, not authoritative.** "Core arts only" is
  BEA's own line 100, and "less the largest three" is a choice made in this notebook and named
  in the configuration cell. Neither is a better definition than BEA's. They exist to measure
  sensitivity, and a different pair would give different numbers.
- **No independent benchmark.** This notebook tests the account against variations of itself.
  It does not compare ACPSA against an external count of arts organisations, employment or
  receipts, so it can show that the answer depends on the boundary without showing which
  boundary is closer to anything.
- **Value added is not activity.** A state can have high arts value added because cultural
  production there is capital-intensive or highly paid, not because more of it happens.
  Exercise 2 exists because this notebook does not resolve that.
- **Revision is unmeasured and, from this interface, unmeasurable.** The API serves one
  vintage and exposes no way to request another. This notebook did not establish BEA's
  revision history for ACPSA, so it can say that a revision would be invisible here — not how
  large one has been.
- **Not tested here:** whether the partial-attribution shares BEA applies to industries like
  retail, construction and government are stable across states, which would determine whether
  a state-to-state comparison of the full account is even comparing the same construct.